# TimeR4 + TEN — TimeQuestions Tiếng Việt

---

## ⚙️ HƯỚNG DẪN SETUP TRƯỚC KHI CHẠY

### Bước 1 — Bật GPU T4
```
Nhìn bên phải → Session options
→ ACCELERATOR: chọn GPU T4 x1
→ Nhấn Save
```

### Bước 2 — Bật Internet
```
Nhìn bên phải → Session options
→ INTERNET: bật toggle → Internet on (màu xanh)
→ Nếu toggle bị khoá → cần xác minh số điện thoại tại kaggle.com/settings
```

### Bước 3 — Chỉnh Persistence
```
Nhìn bên phải → Session options
→ PERSISTENCE: chọn Files only
```

### Bước 4 — Upload 2 file dataset
```
Nhìn bên phải → tab Input → + Add Input → Upload
Upload file 1: TimeQuestions_vi_test.json
Upload file 2: MultiTQ_vi_test.json  (dùng cho TKG)

Sau khi upload, 2 file sẽ hiện trong /kaggle/input/...
```

### Bước 5 — Restart Session
```
Run → Restart Session  (bắt buộc để tránh cache cũ)
```

### Bước 6 — Chạy notebook
```
Nhấn Run All  →  đợi ~1.5-2 giờ
Kết quả xuất hiện ở Cell 7 cuối cùng
Download summary_TQ.json từ tab Output bên phải
```

---

| Cell | Làm gì | Thời gian |
|------|---------|----------|
| 1 | Cài thư viện + load Mistral-7B | ~8 phút |
| 2 | Module TEN + test | ~5 giây |
| 3 | Clone TimeR4 + patch + load data | ~2 phút |
| 4 | Hàm pipeline | ~5 giây |
| 5 | Chạy TimeR4 gốc (không TEN) | ~30 phút |
| 6 | Chạy TimeR4 + TEN | ~30 phút |
| 7 | Bảng kết quả + lưu | ~1 phút |

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — CÀI THƯ VIỆN + LOAD MISTRAL
# ═══════════════════════════════════════════════════════════

import subprocess, sys

# Gỡ thư viện cũ tránh conflict
print('Gỡ thư viện cũ...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
    'peft', 'sentence-transformers', 'transformers',
    'huggingface-hub', 'accelerate', 'triton'],
    capture_output=True)

# Cài đúng version tương thích
pkgs = [
    'huggingface_hub==0.23.4',
    'peft==0.11.1',
    'transformers==4.44.0',
    'sentence-transformers==3.3.1',
    'accelerate==0.34.0',
    'faiss-cpu', 'tqdm', 'spacy',
]
for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print(f'  {"✅" if r.returncode==0 else "❌"} {pkg}')

subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'],
               capture_output=True)
print('  ✅ en_core_web_sm')

# Import tất cả
import torch, os, json, glob, sys, re, copy
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
print('✅ Import OK')

# Kiểm tra GPU
if not torch.cuda.is_available():
    print('⚠️  Không có GPU! Vào Session options → ACCELERATOR → GPU T4')
else:
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

# Load Mistral-7B
MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.2'
print(f'\nLoading {MODEL_NAME}... (~5-8 phút)')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
vram = torch.cuda.memory_allocated()/1e9
print(f'✅ Mistral loaded! VRAM: {vram:.1f} GB')

# Retriever
os.makedirs('/kaggle/working/models', exist_ok=True)
RETRIEVER_PATH = '/kaggle/working/models/multi_model'
if not os.path.exists(RETRIEVER_PATH):
    SentenceTransformer('all-MiniLM-L6-v2').save(RETRIEVER_PATH)
print(f'✅ Retriever: {RETRIEVER_PATH}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — MODULE TEN: 4 PATTERN CỤ THỂ
# ═══════════════════════════════════════════════════════════

def normalize_temporal_expression(question):
    """
    TEN: Chuẩn hóa 4 dạng biểu thức thời gian CỤ THỂ.
    Pattern 1: ngày DD tháng MM năm YYYY  →  YYYY-MM-DD
    Pattern 2: ngày DD/MM/YYYY            →  YYYY-MM-DD
    Pattern 3: tháng MM năm YYYY          →  YYYY-MM
    Pattern 4: tháng MM/YYYY             →  YYYY-MM
    """
    text = question.lower()
    spans = []

    for m in re.finditer(r'ngày\s+(\d{1,2})\s+tháng\s+(\d{1,2})\s+năm\s+(\d{4})', text):
        spans.append((m.start(), m.end(),
                      f'{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}'))

    for m in re.finditer(r'ngày\s+(\d{1,2})/(\d{1,2})/(\d{4})', text):
        spans.append((m.start(), m.end(),
                      f'{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}'))

    for m in re.finditer(r'tháng\s+(\d{1,2})\s+năm\s+(\d{4})', text):
        spans.append((m.start(), m.end(),
                      f'{m.group(2)}-{int(m.group(1)):02d}'))

    for m in re.finditer(r'tháng\s+(\d{1,2})/(\d{4})', text):
        spans.append((m.start(), m.end(),
                      f'{m.group(2)}-{int(m.group(1)):02d}'))

    if not spans: return text

    kept, last_end = [], -1
    for s, e, r in sorted(spans, key=lambda x: -(x[1]-x[0])):
        if s >= last_end:
            kept.append((s, e, r)); last_end = e
    kept.sort(key=lambda x: x[0])
    for s, e, r in reversed(kept):
        text = text[:s] + r + text[e:]
    return text

# Test
tests = [
    (True,  'Trước ngày 22 tháng 10 năm 2008, Malaysia làm gì?'),
    (True,  'Ai ký thỏa thuận với TQ vào tháng 4 năm 2005?'),
    (True,  'Nước nào lên án TQ vào tháng 8/2013?'),
    (True,  'Ai đến Iraq vào ngày 14/10/2015?'),
    (False, 'Jorge luis borges được trao giải gì năm 1971?'),
    (False, 'Cuốn sách đầu tiên Charles Dickens viết là gì?'),
    (False, 'Ai lãnh đạo Anh trong những năm gần đây?'),
]

print('=== TEST MODULE TEN ===')
ok = 0
for should, q in tests:
    r = normalize_temporal_expression(q)
    changed = r != q.lower()
    correct = (changed == should)
    if correct: ok += 1
    print(f'{"✅" if correct else "❌"} [{"normalize" if should else "giữ nguyên"}]: {q[:55]}')
    if changed: print(f'   → {r[:55]}')

print(f'\nKết quả: {ok}/{len(tests)}')
assert ok == len(tests), '❌ TEN test thất bại — DỪNG!'
print('✅ Module TEN OK')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — CLONE TIMER4 + PATCH + LOAD DATA
# ═══════════════════════════════════════════════════════════

if not os.path.exists('/kaggle/working/TimeR4'):
    os.system('git clone --depth 1 https://github.com/qianxinying/TimeR4.git /kaggle/working/TimeR4')
os.chdir('/kaggle/working/TimeR4')

# Patch retrival.py
with open('retrival.py') as f: code = f.read()
patches = [
    ('self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list]',
     'self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list] if id_list is not None else []'),
    ('self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = embedding_size',
     'self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = self.model.get_sentence_embedding_dimension()'),
    ('self.full_time = [datetime.strptime(triple[3], "%Y-%m-%d").date() for triple in triple_list]',
     'self.full_time = []\n        for triple in triple_list:\n            t = triple[3] if len(triple) > 3 else "2000-01-01"\n            if len(t) == 4: t += "-01-01"\n            elif len(t) == 7: t += "-01"\n            try: self.full_time.append(datetime.strptime(t[:10], "%Y-%m-%d").date())\n            except: self.full_time.append(datetime.strptime("2000-01-01", "%Y-%m-%d").date())'),
    ('corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]',
     'if corpus_embeddings.ndim == 1:\n            corpus_embeddings = corpus_embeddings[np.newaxis, :]\n        corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]'),
]
for old, new in patches:
    if old in code: code = code.replace(old, new)
with open('retrival.py', 'w') as f: f.write(code)
if 'retrival' in sys.modules: del sys.modules['retrival']
from retrival import Retrieval
print('✅ retrival.py patched')

# ── Load TimeQuestions-VI ─────────────────────────────────
found_tq = glob.glob('/kaggle/input/**/TimeQuestions_vi_test.json', recursive=True)
if not found_tq:
    print('⚠️  Không tìm thấy TimeQuestions_vi_test.json!')
    print('   Vào tab Input → Add Input → Upload file này')
    raise FileNotFoundError('TimeQuestions_vi_test.json')

TQ_PATH = found_tq[0]
print(f'✅ TimeQuestions-VI: {TQ_PATH}')

with open(TQ_PATH, encoding='utf-8') as f:
    tq_data = json.load(f)

print(f'Tổng: {len(tq_data):,} câu')
print(f'Keys: {list(tq_data[0].keys())}')

# Phân tích
def get_text(item): return item.get('Question', item.get('question', ''))

temporal_q = [q for q in tq_data
              if normalize_temporal_expression(get_text(q)) != get_text(q).lower()]
pct = len(temporal_q)/len(tq_data)*100
print(f'\nCâu CÓ temporal: {len(temporal_q):,}/{len(tq_data):,} ({pct:.1f}%)')

assert pct < 80, f'❌ Coverage {pct:.1f}% > 80% — TEN bắt nhầm! Kiểm tra lại.'

# Lấy câu có temporal để chạy
N_TOTAL   = min(500, len(temporal_q))  # TimeQuestions nhỏ hơn → dùng tối đa 500
questions = temporal_q[:N_TOTAL]
print(f'Dùng: {len(questions)} câu')

print('\nVí dụ 3 câu:')
for q in questions[:3]:
    orig = get_text(q)
    norm = normalize_temporal_expression(orig)
    print(f'  Gốc: {orig}')
    print(f'  →    {norm}')
    print(f'  Ans: {q.get("answers_simple",[])[:2]}')
    print()

# ── Load TKG (dùng MultiTQ TKG vì TimeQuestions không có TKG riêng) ──
# TimeR4 dùng ICEWS TKG cho cả 2 dataset
FACT_PATH   = 'datasets/MultiTQ/kg/full.txt'
triple_list = []
with open(FACT_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip().replace('_', ' ')
        if not line: continue
        parts = line.split('\t') if '\t' in line else line.split()
        if len(parts) >= 3: triple_list.append(parts)
print(f'✅ TKG: {len(triple_list):,} triplets')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — HÀM PIPELINE
# TimeQuestions khác MultiTQ ở trường Question và answer
# ═══════════════════════════════════════════════════════════

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def call_llm(prompt, max_new_tokens=80):
    inp = tokenizer(prompt, return_tensors='pt', truncation=True,
                    max_length=512).to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**inp, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:],
                             skip_special_tokens=True).strip()

def gen_answer(prompt, max_new_tokens=64):
    inp = tokenizer(prompt, return_tensors='pt', truncation=True,
                    max_length=1024).to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**inp, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:],
                             skip_special_tokens=True).strip().split('\n')[0]

def evaluate_hit(predicted, gold):
    """Hit@1 — đúng cho gold dạng list hoặc string."""
    pred = str(predicted).lower().strip()
    # TimeQuestions có answers_simple là list
    golds = gold if isinstance(gold, list) else [gold]
    return any(
        str(g).lower().strip() in pred or pred in str(g).lower().strip()
        for g in golds
    )

def get_q(item):
    """TimeQuestions dùng 'Question' (chữ hoa), MultiTQ dùng 'question'."""
    return item.get('Question', item.get('question', ''))

def get_gold(item):
    """TimeQuestions dùng 'answers_simple'."""
    return item.get('answers_simple',
           item.get('answer', item.get('answers', '?')))

def run_pipeline(qs_raw, triples, use_ten=False, label=''):
    qs = copy.deepcopy(qs_raw)
    print(f'\n{"="*60}')
    print(f'  {label}  TEN={"ON" if use_ten else "OFF"}  n={len(qs)}')
    print(f'{"="*60}')

    q_list = [get_q(q) for q in qs]
    q_norm = [normalize_temporal_expression(q) for q in q_list] if use_ten else q_list

    if use_ten:
        n = sum(1 for a,b in zip(q_list,q_norm) if a.lower()!=b)
        pct = n/len(q_list)*100
        print(f'[TEN] Normalized: {n}/{len(q_list)} ({pct:.1f}%)')
        if pct > 80:
            print('⚠️  WARNING: >80% — kiểm tra lại module TEN!')

    print('[1/4] Retrieve background...')
    r1 = Retrieval(RETRIEVER_PATH, q_norm, triples, None, None)
    d1,c1 = r1.compute_similarity(n=15)
    bg = r1.get_result(d1, c1, q_norm, re_rank=False)

    print('[2/4] Rewrite...')
    for i in tqdm(range(len(qs)), desc='Rewrite', leave=False):
        q = get_q(qs[i])
        f = (bg[i].get('fact') or [''])[0]
        resp = call_llm(
            'Rewrite with explicit time. Output question only.\n'
            f'Fact: {f}\nQuestion: {q}\nRewritten:'
        ).split('\n')[0].strip()
        qs[i]['Question'] = resp or q

    print('[3/4] Retrieve + Rerank...')
    q2 = [get_q(q) for q in qs]
    if use_ten: q2 = [normalize_temporal_expression(q) for q in q2]
    r2 = Retrieval(RETRIEVER_PATH, q2, triples, None, None)
    d2,c2 = r2.compute_similarity(n=15)
    fl = r2.get_result(d2, c2, q2, re_rank=False)

    print('[4/4] Generate...')
    results, correct = [], 0
    for i, item in enumerate(tqdm(qs, desc='Generate', leave=False)):
        q     = get_q(item)
        facts = (fl[i].get('fact') or [])[:3]
        gold  = get_gold(qs_raw[i])
        pred  = gen_answer(
            'Answer the question using the facts. Return only the answer.\n'
            f'Facts: {facts}\nQuestion: {q}\nAnswer:'
        )
        ok = evaluate_hit(pred, gold)
        if ok: correct += 1
        results.append({
            'id':        qs_raw[i].get('Id', i),
            'question':  get_q(qs_raw[i]),
            'q_norm':    q_norm[i] if use_ten else '',
            'gold':      str(gold),
            'predicted': pred,
            'correct':   ok,
            'signal':    qs_raw[i].get('Temporal_signal', []),
            'type':      qs_raw[i].get('Type', ''),
        })

    em = correct/len(results)*100
    print(f'\n✅ {label}: {correct}/{len(results)} = {em:.2f}% Hit@1')
    return results, em

print('✅ Pipeline ready')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — LẦN 1: TimeR4 GỐC (không TEN)
# ═══════════════════════════════════════════════════════════

res_before, em_before = run_pipeline(
    questions, triple_list,
    use_ten=False, label='TimeR4 gốc'
)
with open('/kaggle/working/TQ_before.json', 'w', encoding='utf-8') as f:
    json.dump(res_before, f, ensure_ascii=False, indent=2)
print(f'\n✅ Saved TQ_before.json  |  {em_before:.2f}%')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — LẦN 2: TimeR4 + TEN
# ═══════════════════════════════════════════════════════════

res_after, em_after = run_pipeline(
    questions, triple_list,
    use_ten=True, label='TimeR4 + TEN'
)
with open('/kaggle/working/TQ_after.json', 'w', encoding='utf-8') as f:
    json.dump(res_after, f, ensure_ascii=False, indent=2)
print(f'\n✅ Saved TQ_after.json  |  {em_after:.2f}%')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — BẢNG KẾT QUẢ
# ═══════════════════════════════════════════════════════════

diff      = em_after - em_before
improved  = sum(1 for b,a in zip(res_before,res_after) if a['correct'] and not b['correct'])
degraded  = sum(1 for b,a in zip(res_before,res_after) if b['correct'] and not a['correct'])
unchanged = len(res_before) - improved - degraded
n_norm    = sum(1 for r in res_after if r['q_norm'] and r['q_norm'] != r['question'].lower())

print('═'*65)
print('  BẢNG KẾT QUẢ — TimeQuestions-VI')
print('═'*65)
print(f'  {"Model":<46} {"Hit@1":>8} {"Cải thiện":>9}')
print('  '+'-'*62)
print(f'  {"TimeR4 original (tiếng Anh, paper gốc)":<46} {"42.1%":>8} {"—":>9}')
print('  '+'-'*62)
print(f'  {"TimeR4 baseline (VI-TQ, chưa TEN)":<46} {em_before:>7.2f}% {"—":>9}')
ds = f'+{diff:.2f} ↑' if diff > 0 else f'{diff:.2f}'
print(f'  {"TimeR4 + TEN (VI-TQ, sau normalize)":<46} {em_after:>7.2f}% {ds:>9}')
print('═'*65)
print(f'\n  Dataset:                          TimeQuestions-VI')
print(f'  Số câu test (CÓ temporal):        {len(res_before)}')
print(f'  TEN normalize được:               {n_norm}/{len(res_before)} ({n_norm/len(res_before)*100:.1f}%)')
print(f'  Cải thiện (sai→đúng):             {improved}')
print(f'  Ảnh hưởng (đúng→sai):             {degraded}')
print(f'  Không đổi:                        {unchanged}')

# Ví dụ cải thiện
ex = [(b,a) for b,a in zip(res_before,res_after) if a['correct'] and not b['correct']]
if ex:
    print('\n=== VÍ DỤ CÂU ĐƯỢC CẢI THIỆN ===')
    for b, a in ex[:3]:
        print(f'  Q:     {b["question"]}')
        if a['q_norm'] and a['q_norm'] != b['question'].lower():
            print(f'  →      {a["q_norm"]}')
        print(f'  Gold:  {str(b["gold"])[:80]}')
        print(f'  Trước: {b["predicted"][:60]}  ❌')
        print(f'  Sau:   {a["predicted"][:60]}  ✅')
        print()
else:
    print('\n(Không có câu nào được cải thiện)')

summary = {
    'dataset': 'TimeQuestions-VI',
    'n': len(res_before), 'em_paper': 42.1,
    'em_before': round(em_before,2),
    'em_after': round(em_after,2),
    'diff': round(diff,2),
    'coverage': f'{n_norm}/{len(res_before)} ({n_norm/len(res_before)*100:.1f}%)',
    'improved': improved, 'degraded': degraded, 'unchanged': unchanged,
}
with open('/kaggle/working/summary_TQ.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print('\n✅ summary_TQ.json saved → download từ tab Output bên phải')